# PathSnap — Dataset Exploration (EDA)

This notebook performs dataset exploration for the PathSnap map-matching project.

**Important**
- Uses only the `train` and `student_test` partitions.
- Does **not** open or inspect the evaluator-testing partition.
- Does not assume GPS observations correspond one-to-one with route labels.
- Computes physical GPS distances using WGS84 geodesic calculations, not Euclidean distance in longitude/latitude degrees.
- Saves tables and visualizations under `PathSnap/results/eda/`.

Run the cells from top to bottom in Google Colab. The notebook reuses an existing Google Drive mount when available and otherwise mounts Drive at `/content/gdrive` to avoid conflicts with an already-populated `/content/drive` folder.


In [ ]:
# 1. Google Drive connection and project paths

from pathlib import Path

if Path("/content/drive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive")
elif Path("/content/gdrive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")
else:
    from google.colab import drive
    drive.mount("/content/gdrive")
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

PROJECT_DIR = DRIVE_ROOT / "PathSnap"
EDA_DIR = PROJECT_DIR / "results" / "eda"
VIZ_DIR = EDA_DIR / "visualizations"

EDA_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Project directory:", PROJECT_DIR)
print("EDA output directory:", EDA_DIR)


In [ ]:
# 2. Install/import required libraries

%pip -q install networkx folium pyproj

import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import folium

from IPython.display import display
from pyproj import Geod

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

WGS84 = Geod(ellps="WGS84")


In [ ]:
# 3. Locate the extracted dataset automatically

# Expected structure from the supplied dataset:
# PathSnap/data/student_dataset/Student folder/
#     train/<trip_id>/
#     student_test/<trip_id>/
#
# This cell also checks a common data/raw location.

possible_roots = [
    PROJECT_DIR / "data" / "student_dataset" / "Student folder",
    PROJECT_DIR / "data" / "raw" / "student_dataset" / "Student folder",
    PROJECT_DIR / "data" / "student_dataset",
    PROJECT_DIR / "data" / "raw" / "student_dataset",
]

DATA_DIR = None
for candidate in possible_roots:
    if (candidate / "train").is_dir() and (candidate / "student_test").is_dir():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    search_base = PROJECT_DIR / "data"
    if search_base.exists():
        for train_dir in search_base.rglob("train"):
            parent = train_dir.parent
            if (parent / "student_test").is_dir():
                DATA_DIR = parent
                break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate folders named 'train' and 'student_test'. "
        "Check that the dataset ZIP has been extracted under PathSnap/data/."
    )

print("Dataset directory:", DATA_DIR)
print("Train directory exists:", (DATA_DIR / "train").is_dir())
print("Student-test directory exists:", (DATA_DIR / "student_test").is_dir())


In [ ]:
# 4. Dataset file-loading helpers

def read_whitespace_file(path, names):
    """Read a whitespace-separated dataset file without a header."""
    if not path.exists():
        return None
    return pd.read_csv(path, sep=r"\\s+", header=None, names=names, engine="python")

def load_trip(trip_folder):
    trip_id = trip_folder.name
    return {
        "track": read_whitespace_file(
            trip_folder / f"{trip_id}.track",
            ["longitude", "latitude", "timestamp"]
        ),
        "nodes": read_whitespace_file(
            trip_folder / f"{trip_id}.nodes",
            ["longitude", "latitude"]
        ),
        "arcs": read_whitespace_file(
            trip_folder / f"{trip_id}.arcs",
            ["from_node", "to_node"]
        ),
        "route": read_whitespace_file(
            trip_folder / f"{trip_id}.route",
            ["arc_id"]
        ),
    }

def trip_folders(partition):
    folder = DATA_DIR / partition
    if not folder.is_dir():
        return []
    return sorted(p for p in folder.iterdir() if p.is_dir())

for part in ["train", "student_test"]:
    print(f"{part}: {len(trip_folders(part))} trip folders")


In [ ]:
# 5. Overall dataset exploration

overall_rows = []

for partition in ["train", "student_test"]:
    for trip_folder in trip_folders(partition):
        trip_id = trip_folder.name
        files = {
            "track": trip_folder / f"{trip_id}.track",
            "nodes": trip_folder / f"{trip_id}.nodes",
            "arcs": trip_folder / f"{trip_id}.arcs",
            "route": trip_folder / f"{trip_id}.route",
        }
        data = load_trip(trip_folder)
        track, nodes, arcs, route = (
            data["track"], data["nodes"], data["arcs"], data["route"]
        )

        row = {
            "partition": partition,
            "trip_id": trip_id,
            "track_file_present": files["track"].exists(),
            "nodes_file_present": files["nodes"].exists(),
            "arcs_file_present": files["arcs"].exists(),
            "route_file_present": files["route"].exists(),
            "gps_points": len(track) if track is not None else np.nan,
            "road_nodes": len(nodes) if nodes is not None else np.nan,
            "road_arcs": len(arcs) if arcs is not None else np.nan,
            "route_segments": len(route) if route is not None else np.nan,
            "track_file_kb": round(files["track"].stat().st_size / 1024, 2) if files["track"].exists() else np.nan,
            "nodes_file_kb": round(files["nodes"].stat().st_size / 1024, 2) if files["nodes"].exists() else np.nan,
            "arcs_file_kb": round(files["arcs"].stat().st_size / 1024, 2) if files["arcs"].exists() else np.nan,
            "route_file_kb": round(files["route"].stat().st_size / 1024, 2) if files["route"].exists() else np.nan,
        }

        if track is not None:
            row["missing_gps_cells"] = int(track.isna().sum().sum())
            row["invalid_coordinate_rows"] = int((
                ~track["longitude"].between(-180, 180) |
                ~track["latitude"].between(-90, 90)
            ).sum())
            row["timestamps_monotonic"] = bool(track["timestamp"].is_monotonic_increasing)
        else:
            row["missing_gps_cells"] = np.nan
            row["invalid_coordinate_rows"] = np.nan
            row["timestamps_monotonic"] = False

        overall_rows.append(row)

eda_df = pd.DataFrame(overall_rows)
display(eda_df.head(10))
print("Total inspected trips:", len(eda_df))


In [ ]:
# 6. Partition-level summary

partition_summary = (
    eda_df.groupby("partition", dropna=False)
    .agg(
        trips=("trip_id", "count"),
        total_gps_points=("gps_points", "sum"),
        total_road_nodes=("road_nodes", "sum"),
        total_road_arcs=("road_arcs", "sum"),
        total_route_segments=("route_segments", "sum"),
        trips_with_missing_gps=("missing_gps_cells", lambda s: int((s.fillna(0) > 0).sum())),
        trips_with_invalid_coordinates=("invalid_coordinate_rows", lambda s: int((s.fillna(0) > 0).sum())),
    )
    .reset_index()
)

display(partition_summary)


In [ ]:
# 7. GPS quality, timestamp gaps, distance and speed analysis

gps_rows = []

for partition in ["train", "student_test"]:
    for trip_folder in trip_folders(partition):
        data = load_trip(trip_folder)
        track = data["track"]
        if track is None or track.empty:
            continue

        track = track.copy()
        track = track.dropna(subset=["longitude", "latitude", "timestamp"])

        lon = track["longitude"].to_numpy(dtype=float)
        lat = track["latitude"].to_numpy(dtype=float)
        ts = track["timestamp"].to_numpy(dtype=float)

        time_diff = np.diff(ts)
        valid_coord = (
            np.isfinite(lon) & np.isfinite(lat) &
            (lon >= -180) & (lon <= 180) &
            (lat >= -90) & (lat <= 90)
        )

        distance_m = np.array([])
        speed_mps = np.array([])
        if len(lon) >= 2 and valid_coord.all():
            _, _, distance_m = WGS84.inv(lon[:-1], lat[:-1], lon[1:], lat[1:])
            distance_m = np.asarray(distance_m, dtype=float)
            positive_dt = time_diff > 0
            speed_mps = np.divide(
                distance_m, time_diff,
                out=np.full_like(distance_m, np.nan, dtype=float),
                where=positive_dt
            )

        gps_rows.append({
            "partition": partition,
            "trip_id": trip_folder.name,
            "gps_points": len(track),
            "missing_cells": int(track.isna().sum().sum()),
            "invalid_coordinate_rows": int((~valid_coord).sum()),
            "timestamp_monotonic": bool(pd.Series(ts).is_monotonic_increasing),
            "duplicate_timestamps": int(pd.Series(ts).duplicated().sum()),
            "mean_time_gap_s": float(np.mean(time_diff)) if len(time_diff) else np.nan,
            "median_time_gap_s": float(np.median(time_diff)) if len(time_diff) else np.nan,
            "max_time_gap_s": float(np.max(time_diff)) if len(time_diff) else np.nan,
            "gaps_over_2s": int((time_diff > 2).sum()),
            "negative_or_zero_time_gaps": int((time_diff <= 0).sum()),
            "longitude_min": float(np.min(lon)) if len(lon) else np.nan,
            "longitude_max": float(np.max(lon)) if len(lon) else np.nan,
            "latitude_min": float(np.min(lat)) if len(lat) else np.nan,
            "latitude_max": float(np.max(lat)) if len(lat) else np.nan,
            "gps_path_distance_km": float(np.nansum(distance_m) / 1000) if len(distance_m) else np.nan,
            "median_speed_mps": float(np.nanmedian(speed_mps)) if len(speed_mps) else np.nan,
            "max_speed_mps": float(np.nanmax(speed_mps)) if len(speed_mps) and np.isfinite(speed_mps).any() else np.nan,
        })

gps_results = pd.DataFrame(gps_rows)
display(gps_results.head(10))


In [ ]:
# 8. Road-network structure and graph connectivity

network_rows = []

for partition in ["train", "student_test"]:
    for trip_folder in trip_folders(partition):
        data = load_trip(trip_folder)
        nodes, arcs = data["nodes"], data["arcs"]

        if nodes is None or arcs is None or nodes.empty:
            continue

        n_nodes = len(nodes)
        arc_from = arcs["from_node"].to_numpy(dtype=int)
        arc_to = arcs["to_node"].to_numpy(dtype=int)

        endpoint_valid = (
            (arc_from >= 0) & (arc_from < n_nodes) &
            (arc_to >= 0) & (arc_to < n_nodes)
        )

        G = nx.DiGraph()
        G.add_nodes_from(range(n_nodes))
        valid_edges = list(zip(arc_from[endpoint_valid], arc_to[endpoint_valid]))
        G.add_edges_from(valid_edges)

        weak_components = list(nx.weakly_connected_components(G)) if G.number_of_nodes() else []
        strong_components = list(nx.strongly_connected_components(G)) if G.number_of_nodes() else []
        largest_weak = max((len(c) for c in weak_components), default=0)

        network_rows.append({
            "partition": partition,
            "trip_id": trip_folder.name,
            "node_count": n_nodes,
            "arc_count": len(arcs),
            "valid_arc_endpoints": int(endpoint_valid.sum()),
            "invalid_arc_endpoints": int((~endpoint_valid).sum()),
            "self_loop_arcs": int((arc_from == arc_to).sum()),
            "directed_graph_edges": G.number_of_edges(),
            "isolated_nodes": len(list(nx.isolates(G))),
            "weakly_connected_components": len(weak_components),
            "strongly_connected_components": len(strong_components),
            "largest_weak_component_nodes": largest_weak,
            "largest_weak_component_fraction": largest_weak / n_nodes if n_nodes else np.nan,
        })

network_results = pd.DataFrame(network_rows)
display(network_results.head(10))


In [ ]:
# 9. Route-label validity and sequential continuity checks

route_rows = []

for partition in ["train", "student_test"]:
    for trip_folder in trip_folders(partition):
        data = load_trip(trip_folder)
        arcs, route = data["arcs"], data["route"]

        if arcs is None or route is None or route.empty:
            continue

        route_ids_raw = route["arc_id"]
        route_ids_numeric = pd.to_numeric(route_ids_raw, errors="coerce")
        ids_are_integer = route_ids_numeric.notna().all() and np.all(
            np.equal(route_ids_numeric.to_numpy(), np.floor(route_ids_numeric.to_numpy()))
        )

        if ids_are_integer:
            route_ids = route_ids_numeric.astype(int).to_numpy()
            in_range = (route_ids >= 0) & (route_ids < len(arcs))
            invalid_count = int((~in_range).sum())

            valid_ids = route_ids[in_range]
            selected_arcs = arcs.iloc[valid_ids] if len(valid_ids) else arcs.iloc[0:0]

            continuity_breaks = 0
            if len(selected_arcs) >= 2:
                from_nodes = selected_arcs["from_node"].to_numpy()
                to_nodes = selected_arcs["to_node"].to_numpy()
                continuity_breaks = int((to_nodes[:-1] != from_nodes[1:]).sum())
        else:
            invalid_count = len(route)
            continuity_breaks = np.nan
            in_range = np.array([], dtype=bool)

        route_rows.append({
            "partition": partition,
            "trip_id": trip_folder.name,
            "route_length": len(route),
            "route_ids_integer": bool(ids_are_integer),
            "invalid_arc_id_count": invalid_count,
            "all_route_arc_ids_in_range": bool(ids_are_integer and invalid_count == 0),
            "route_continuity_breaks": continuity_breaks,
            "route_is_continuous": bool(ids_are_integer and invalid_count == 0 and continuity_breaks == 0),
        })

route_results = pd.DataFrame(route_rows)
display(route_results.head(10))


In [ ]:
# 10. Visual summaries

# Trips per partition
trip_counts = eda_df.groupby("partition")["trip_id"].count()
plt.figure(figsize=(6, 4))
trip_counts.plot(kind="bar")
plt.title("Number of trips by partition")
plt.xlabel("Partition")
plt.ylabel("Number of trips")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(VIZ_DIR / "trips_by_partition.png", dpi=160)
plt.show()

# GPS points per trip
plt.figure(figsize=(10, 4))
for partition in ["train", "student_test"]:
    subset = eda_df[eda_df["partition"] == partition]
    plt.hist(subset["gps_points"].dropna(), bins=20, alpha=0.6, label=partition)
plt.title("GPS observation counts per trip")
plt.xlabel("GPS points")
plt.ylabel("Number of trips")
plt.legend()
plt.tight_layout()
plt.savefig(VIZ_DIR / "gps_points_distribution.png", dpi=160)
plt.show()

# Route length per trip
plt.figure(figsize=(10, 4))
for partition in ["train", "student_test"]:
    subset = route_results[route_results["partition"] == partition]
    plt.hist(subset["route_length"].dropna(), bins=20, alpha=0.6, label=partition)
plt.title("Route arc counts per trip")
plt.xlabel("Route segments (arc IDs)")
plt.ylabel("Number of trips")
plt.legend()
plt.tight_layout()
plt.savefig(VIZ_DIR / "route_length_distribution.png", dpi=160)
plt.show()


In [ ]:
# 11. Folium map of one training trip (GPS trace and its road network)

train_folders = trip_folders("train")
if not train_folders:
    print("No training trips available for map preview.")
else:
    sample_folder = train_folders[0]
    sample = load_trip(sample_folder)
    track, nodes, arcs = sample["track"], sample["nodes"], sample["arcs"]

    if track is not None and not track.empty:
        map_track = track.dropna(subset=["longitude", "latitude"]).copy()
        map_track = map_track[
            map_track["longitude"].between(-180, 180) &
            map_track["latitude"].between(-90, 90)
        ]

        if not map_track.empty:
            center = [map_track["latitude"].mean(), map_track["longitude"].mean()]
            route_map = folium.Map(location=center, zoom_start=13)

            gps_points = map_track[["latitude", "longitude"]].to_numpy().tolist()
            folium.PolyLine(
                gps_points,
                color="blue",
                weight=3,
                opacity=0.8,
                tooltip="Observed GPS trace"
            ).add_to(route_map)

            folium.Marker(
                gps_points[0],
                tooltip="GPS start",
                icon=folium.Icon(color="green")
            ).add_to(route_map)
            folium.Marker(
                gps_points[-1],
                tooltip="GPS end",
                icon=folium.Icon(color="red")
            ).add_to(route_map)

            # Draw the labelled route from route arc IDs, when available and valid.
            route_data = sample["route"]
            if nodes is not None and arcs is not None and route_data is not None:
                route_ids = pd.to_numeric(route_data["arc_id"], errors="coerce")
                if route_ids.notna().all():
                    route_ids = route_ids.astype(int).to_numpy()
                    if len(route_ids) and ((route_ids >= 0) & (route_ids < len(arcs))).all():
                        selected = arcs.iloc[route_ids]
                        node_ids = [int(selected.iloc[0]["from_node"])] + selected["to_node"].astype(int).tolist()
                        if all(0 <= node_id < len(nodes) for node_id in node_ids):
                            route_coords = [
                                [float(nodes.iloc[node_id]["latitude"]),
                                 float(nodes.iloc[node_id]["longitude"])]
                                for node_id in node_ids
                            ]
                            folium.PolyLine(
                                route_coords,
                                color="orange",
                                weight=4,
                                opacity=0.8,
                                tooltip="Ground-truth route from route arc IDs"
                            ).add_to(route_map)

            folium.LayerControl().add_to(route_map)
            map_path = VIZ_DIR / f"folium_sample_trip_{sample_folder.name}.html"
            route_map.save(str(map_path))
            print("Map saved:", map_path)
            display(route_map)
        else:
            print("Sample training trip has no valid GPS coordinates to map.")


In [ ]:
# 12. Save all EDA tables to Google Drive

tables = {
    "dataset_eda.csv": eda_df,
    "partition_summary.csv": partition_summary,
    "gps_analysis.csv": gps_results,
    "network_analysis.csv": network_results,
    "route_validation.csv": route_results,
}

for filename, dataframe in tables.items():
    output_path = EDA_DIR / filename
    dataframe.to_csv(output_path, index=False)
    print(f"Saved: {output_path} ({len(dataframe)} rows)")

print("\\nEDA export completed.")


In [ ]:
# 13. Verify saved outputs

for path in sorted(EDA_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(PROJECT_DIR)}  |  {path.stat().st_size / 1024:.2f} KB")


## Interpretation and limitations

- `dataset_eda.csv` describes file availability, counts, and basic GPS data-quality checks.
- `gps_analysis.csv` describes timestamp intervals, GPS coordinate ranges, geodesic path distance, and approximate point-to-point speeds.
- `network_analysis.csv` describes graph structure and connectivity for each trip's road network.
- `route_validation.csv` checks route arc IDs against the `.arcs` table and checks whether consecutive route arcs connect (`to_node` of one arc equals `from_node` of the next).
- The Folium HTML map is a visual inspection aid. It overlays the observed GPS trace and, when valid, the route reconstructed from the route arc IDs.
- The `.route` sequence is an arc-ID sequence. It must **not** be treated as a row-by-row label for GPS observations.
- EDA results do not establish model performance. Model training and evaluation must be performed separately, with trip-level validation and without using evaluator-test labels during development.
